In [1]:
import pandas as pd
import yfinance as yf

In [2]:
#dictionary of global indices
indices = {
    "^GSPC": "S&P 500",
    "^IXIC": "NASDAQ",
    "^DJI": "Dow Jones",
    "^NSEI": "Nifty 50",
    "^BSESN": "Sensex"
}

In [4]:
# function for downloading and transform
def download_index(ticker,index_name,start_date,end_date):
    df = yf.download(
        ticker,
        start =start_date,
        end = end_date
    )

    #flattenning the multiindex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    #removing the column indexx name
    df.columns.name = None

    #convert the date index index to normal column
    df.reset_index(inplace=True)

    #add the business metric
    df["Ticker"] = ticker
    df["IndexName"] = index_name

    df = df[
    [
        "Date",
        "Ticker",
        "IndexName",
        "Open",
        "High",
        "Low",
        "Close",
        "Volume"
    ]
    ]
    return df 

all_data = []

for ticker, index_name in indices.items():
    print(f"Downloading {index_name}...")

    df = download_index(
        ticker=ticker,
        index_name=index_name,
        start_date="2020-01-01",
        end_date="2026-07-18",
    )

    all_data.append(df)

global_index_df = pd.concat(all_data, ignore_index=True)

[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


[*********************100%***********************]  1 of 1 completed


In [5]:
#validation
global_index_df = pd.concat(all_data, ignore_index=True)

In [6]:
#Shape
print("\n Shape")
print(global_index_df.shape)


 Shape
(8161, 8)


In [7]:
#datatypes
print("\nDatatypes")
print(global_index_df.dtypes)


Datatypes
Date         datetime64[ns]
Ticker               object
IndexName            object
Open                float64
High                float64
Low                 float64
Close               float64
Volume                int64
dtype: object


In [8]:
#checking the missing values
print("\nMissing Values:")
print(global_index_df.isnull().sum())


Missing Values:
Date         0
Ticker       0
IndexName    0
Open         0
High         0
Low          0
Close        0
Volume       0
dtype: int64


In [9]:
#checking the duplicat rows
print("\nDuplicated Values")
print(global_index_df.duplicated().sum())


Duplicated Values
0


In [13]:
global_index_df.tail(100)

,Date,Ticker,IndexName,Open,High,Low,Close,Volume
8061,2026-02-18,^BSESN,Sensex,83553.593750,83770.046875,83163.617188,83734.250000,20800
8062,2026-02-19,^BSESN,Sensex,83969.820312,83979.359375,82264.203125,82498.140625,13400
8063,2026-02-20,^BSESN,Sensex,82272.492188,83132.078125,82206.210938,82814.710938,19600
8064,2026-02-23,^BSESN,Sensex,82906.828125,83486.148438,82906.828125,83294.656250,20300
8065,2026-02-24,^BSESN,Sensex,83052.539062,83079.507812,81934.726562,82225.921875,20200
...,...,...,...,...,...,...,...,...
8156,2026-07-13,^BSESN,Sensex,76963.351562,77789.289062,76857.429688,77616.398438,17000
8157,2026-07-14,^BSESN,Sensex,77272.343750,77402.789062,77001.476562,77054.937500,15700
8158,2026-07-15,^BSESN,Sensex,77192.757812,77646.273438,76982.820312,77185.429688,11400
8159,2026-07-16,^BSESN,Sensex,77388.421875,77579.687500,77086.421875,77186.867188,25600


In [14]:
global_index_df.rename(
    columns={
        "Date": "TradeDate",
        "Open": "OpenPrice",
        "High": "HighPrice",
        "Low": "LowPrice",
        "Close": "ClosePrice",
    },
    inplace=True,
)

In [15]:
print(global_index_df.columns)

Index(['TradeDate', 'Ticker', 'IndexName', 'OpenPrice', 'HighPrice',
       'LowPrice', 'ClosePrice', 'Volume'],
      dtype='object')


In [17]:
from sqlalchemy import create_engine
import urllib

In [20]:
from sqlalchemy import create_engine
import urllib

server = "ASTHA"
database = "GlobalIndexAnalytics"

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

connection_url = (
    "mssql+pyodbc:///?odbc_connect="
    + urllib.parse.quote_plus(connection_string)
)

engine = create_engine(connection_url)

In [21]:
with engine.connect() as conn:
    print("Connection Successful!")

Connection Successful!


C:\Users\nagat\AppData\Local\Temp\ipykernel_15588\2754817518.py:1: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [23]:
global_index_df.to_sql(
    name="GlobalIndexPrices",
    con=engine,
    if_exists="append",
    index=False,
)

IntegrityError: (pyodbc.IntegrityError) ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Violation of UNIQUE KEY constraint 'UQ_GlobalIndexPrices'. Cannot insert duplicate key in object 'dbo.GlobalIndexPrices'. The duplicate key value is (2020-01-02, ^GSPC). (2627) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")
[SQL: INSERT INTO [GlobalIndexPrices] ([TradeDate], [Ticker], [IndexName], [OpenPrice], [HighPrice], [LowPrice], [ClosePrice], [Volume]) VALUES (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ? ... 6598 characters truncated ... , ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?), (?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: (datetime.datetime(2020, 1, 2, 0, 0), '^GSPC', 'S&P 500', 3244.669921875, 3258.139892578125, 3235.530029296875, 3257.85009765625, 3459930000, datetime.datetime(2020, 1, 3, 0, 0), '